In [2]:
import networkx as nx
import pandas as pd

% \paragraph{node degree} results saved in node degree check
% considering the ppi-embs learned from string can capture node degree information, and it is known that in biological network, hub genes with high node degrees have higher chance to be defined as disease related genes, there's a bias that model just learned the  node degree from PPI-embs, regardless the given disease-genes (positive labels). Based on that,  we carried experiments to check if the current results are same with prediction with only node degree as input. we calculated genes' node degree from STRING network, for each disease, applied Log + Min–Max normalization to scale node degrees to test genes' node degree to scale [0,1], meanwhile Compresses hubs, Preserves ranking. take this score as prediction values, later using the same metrics to evalute the performance only use node degree as input.

In [3]:
edge_path= '/itf-fi-ml/shared/users/ziyuzh/svm/data/stringdb/edge_2019.csv'
df = pd.read_csv(edge_path)

G = nx.from_pandas_edgelist(df, 'p1', 'p2')
degree_dict = dict(G.degree())

In [9]:
from rdkit.ML.Scoring.Scoring import CalcBEDROC
from sklearn.metrics import roc_auc_score
import numpy as np
import os
import pickle

root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/'

paths = ['df_gnn_occsvm_pred']
out_path_pred = '/itf-fi-ml/shared/users/ziyuzh/svm/results/node_degree_check'
os.makedirs(out_path_pred, exist_ok=True)

def top_recall(cut, y_scores, y_test):
    total_pos = np.sum(y_test == 1)
    if total_pos == 0:
        return 0.0
    top_idx = np.argsort(y_scores)[-cut:][::-1]
    TP = np.sum(y_test[top_idx] == 1)
    return TP / total_pos

result_df = pd.DataFrame(columns=['disease','setting','method','top_recall_150','succ_150','w_recall_150','top_recall_300','succ_300','w_recall_300','bedroc_150','bedroc_300','auroc'])
all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/dga_time_uniport.csv')

for filename in os.listdir(os.path.join(root,paths[0])):
    disease = filename[:9]
    node_degree_prediction = dict()
    for path in paths:
        with open(os.path.join(root, path,filename), "rb") as f:
            data = pickle.load(f)
        y_label = data['true_label']

        test_gene_order = data['test_genes']
        deg = np.array([degree_dict.get(g, 0) for g in test_gene_order])
        deg_log = np.log1p(deg)  # log(1 + degree)
        y_pred = (deg_log - deg_log.min()) / (deg_log.max() - deg_log.min())
        node_degree_prediction['true_label'] = y_label
        node_degree_prediction['test_genes'] = test_gene_order
        node_degree_prediction['node_degree_pred'] = y_pred
        node_degree_prediction['train_pos_genes'] = data['train_pos_genes']

        scores = np.column_stack((y_label, y_pred))  # Stack labels and scores as columns
        scores = scores[scores[:, 1].argsort()[::-1]]

        top_idx = np.argsort(y_pred)[::-1]
        res = []
        for threshold in [150,300]:
            succeed_rate = int(np.sum(y_label[top_idx][:threshold] == 1) > 0)

            true_positive_genes = data['test_genes'][y_label == 1]
            subdf = all_df[(all_df['disease_id']==disease)&(all_df['string_id']).isin(true_positive_genes)][['score','string_id']]
            if len(subdf) != len(true_positive_genes):
                print('waring',subdf,disease)
                break
            score_map = dict(zip(subdf['string_id'], subdf['score']))

            # Create new array with matching scores
            new_scores = np.array([score_map.get(gene, 0) for gene in data['test_genes']])
            weighted_recall = np.sum(new_scores[top_idx][:threshold])/subdf['score'].sum()

            res.extend([top_recall(threshold,y_pred,y_label),succeed_rate,weighted_recall])
        row = [disease, path, 'node_degree'] + list(res)
        row.extend([CalcBEDROC(scores, col=0, alpha=160.9),CalcBEDROC(scores, col=0, alpha=80.45),roc_auc_score(y_label, y_pred)])

        result_df.loc[len(result_df)] = row

    pred_path = os.path.join(out_path_pred, f'{disease}_pred.pkl')
    with open(pred_path, 'wb') as f:
        pickle.dump(node_degree_prediction, f)


In [10]:
node_degree_prediction.keys()

dict_keys(['true_label', 'test_genes', 'node_degree_pred', 'train_pos_genes'])

In [7]:
result_df.groupby('method').mean(numeric_only=True).round(4)

,top_recall_150,succ_150,w_recall_150,top_recall_300,succ_300,w_recall_300,bedroc_150,bedroc_300,auroc
method,,,,,,,,,
node_degree,0.0525,0.1875,0.0587,0.0648,0.2708,0.071,0.0395,0.0554,0.6634
